# RAG Knowledge Bases

Wintermute's **RAG (Retrieval-Augmented Generation)** system augments AI responses with
domain-specific documents. Knowledge bases are directories of documents (PDF, TXT, MD)
that get indexed into vector embeddings and registered as queryable LLM providers.

This notebook covers:
1. Knowledge base directory structure
2. Configuration via `rag_config.json`
3. Bootstrapping and indexing
4. Dynamic RAG querying with `router.set_default()`
5. Tools JSON — linking tools to RAG knowledge bases and MCPs
6. Console integration

## Directory Structure

Knowledge bases live under a base directory (default: `knowledge_bases/`). Each subdirectory
is a separate knowledge base with its own configuration and documents:

```
knowledge_bases/
└── <kb_name>/
    ├── rag_config.json       # Required: provider and embedding config
    ├── storage_db/           # Auto-generated: vector index storage
    │   ├── docstore.json
    │   ├── index_store.json
    │   └── default__vector_store.json
    └── *.pdf, *.txt, *.md    # Source documents to index
```

In [ ]:
import json
from pathlib import Path

kb_base = Path("knowledge_bases")
if kb_base.exists():
    for kb_dir in sorted(kb_base.iterdir()):
        if kb_dir.is_dir():
            config_path = kb_dir / "rag_config.json"
            print(f"Knowledge Base: {kb_dir.name}")
            if config_path.exists():
                config = json.loads(config_path.read_text())
                print(f"  Config: {json.dumps(config, indent=4)}")
            storage = kb_dir / "storage_db"
            if storage.exists():
                print(f"  Index files: {[f.name for f in storage.iterdir()]}")
            docs = [
                f.name for f in kb_dir.iterdir() if f.suffix in (".pdf", ".txt", ".md")
            ]
            if docs:
                print(f"  Documents: {docs}")
            print()
else:
    print("Run from examples/ directory to see knowledge bases")

## rag_config.json

Each knowledge base requires a `rag_config.json` with three fields:

| Field | Description | Example |
|-------|-------------|--------|
| `base_provider_id` | LLM provider for generation (answering) | `"bedrock"`, `"openai"`, `"groq"` |
| `embed_provider_id` | Provider for creating embeddings | `"local_embedder"` |
| `embed_model_id` | Specific embedding model to use | `"BAAI/bge-small-en-v1.5"` |

In [ ]:
import json

config_template = {
    "base_provider_id": "bedrock",
    "embed_provider_id": "local_embedder",
    "embed_model_id": "BAAI/bge-small-en-v1.5",
}

print("rag_config.json template:")
print(json.dumps(config_template, indent=2))
print()
print("Supported embed providers:")
print("  local_embedder — HuggingFace sentence-transformers (runs locally)")
print()
print("Supported base providers:")
print("  bedrock  — AWS Bedrock (Claude, Titan, etc.)")
print("  openai   — OpenAI (GPT-4, etc.)")
print("  groq     — Groq (fast inference)")

## Bootstrapping RAG

`bootstrap_rags()` scans the knowledge base directories, builds vector indices from source
documents, and registers each knowledge base as a named provider in the `llms` registry.

Once bootstrapped, knowledge bases are accessible as `rag-<kb_name>` providers.

In [ ]:
from wintermute.ai.provider import llms

# bootstrap_rags scans knowledge base directories, builds vector indices,
# and registers each KB as a named LLM provider.
#
# Usage (requires dependencies: llama-index, sentence-transformers):
# rag_providers = bootstrap_rags(llms)
# print(f"RAG providers: {llms.providers()}")

print("bootstrap_rags(registry: LLMRegistry) -> list[RAGProvider]")
print()
print("Workflow:")
print("  1. Scans knowledge_bases/ and external_repos/ for storage_db dirs")
print("  2. Loads existing vector indices (or builds new ones from documents)")
print("  3. Creates RAGProvider instances with configured embed/base models")
print("  4. Registers each as 'rag-<kb_name>' in the LLM registry")
print()
print("Environment variables:")
print("  DEFAULT_RAG_PROVIDER  — base LLM for generation")
print("  DEFAULT_EMBED_PROVIDER — embedding provider")
print("  DEFAULT_EMBED_MODEL   — embedding model ID")

## Dynamic RAG Querying with `router.set_default()`

The **Router** is the central dispatch for all LLM requests. Its `set_default()` method
controls which provider handles queries:

```python
router.set_default(*, provider: str | None = None, model: str | None = None) -> None
```

When you call `router.set_default(provider="bedrock")`, subsequent calls to
`router.choose(req)` will route to the Bedrock provider directly — the LLM answers
from its training data alone, with no document augmentation.

When you call `router.set_default(provider="rag-tiny_hardware_test")`, the router
switches to the RAG provider. The RAG provider:
1. Takes the user's query and searches the vector index for relevant document chunks
2. Prepends the retrieved context to the prompt
3. Forwards the augmented prompt to its underlying `base_provider` (e.g., Bedrock)
4. Returns the context-aware response

This means `set_default()` acts as a **live switch** — you can toggle between
base-only and RAG-augmented responses at any point during a session.

### Prerequisites

To run the example below you need:
- **AWS credentials** configured (for Bedrock) or API keys for your chosen provider
- **`knowledge_bases/tiny_hardware_test/`** with a pre-built `storage_db/` index
- **Dependencies**: `llama-index`, `sentence-transformers`, `boto3`
- **Environment variables**: `AWS_REGION`, optionally `BEDROCK_MODEL_ID`, `GROQ_API_KEY`, `OPENAI_API_KEY`
- Run from the `examples/` directory so `knowledge_bases/` is discoverable

In [ ]:
import logging

from wintermute.ai.bootstrap import init_router
from wintermute.ai.types import ChatRequest, Message

logging.basicConfig(level=logging.INFO)


def test_dynamic_rag():
    print("--- 1. Bootstrapping Wintermute ---")
    # This automatically registers your base LLMs and scans for your RAG folders
    router = init_router()

    available_providers = llms.providers()
    print(f"\n[Registry Check] Available Providers: {available_providers}")

    # Prepare the query using your actual types
    query = (
        "What voltage does the Wintermute Quantum Processor use on the VCC_CORE pin? "
        "Also, where are the JTAG pins?"
    )
    req = ChatRequest(messages=[Message(role="user", content=query)])

    # ==========================================
    # TEST 1: NO RAG (Direct to Bedrock/Base)
    # ==========================================
    print("\n--- 2. Test: WITHOUT RAG (Naked Provider) ---")
    router.set_default(provider="bedrock")  # Bypassing RAG entirely

    # The router chooses the provider and prepares the request based on its policy
    base_provider, routed_base_req = router.choose(req)

    try:
        base_resp = base_provider.chat(routed_base_req)
        print(f"Wintermute (Base): {base_resp.content}")
    except Exception as e:
        print(f"Base Query Failed: {e}")

    # ==========================================
    # TEST 2: WITH RAG
    # ==========================================
    print("\n--- 3. Test: WITH RAG (Hardware Oracle) ---")
    rag_target = "rag-tiny_hardware_test"

    if rag_target not in available_providers:
        print(
            f"Error: {rag_target} was not discovered. Check your knowledge_bases folder!"
        )
        return

    # Tell the router to point at the newly discovered RAG provider
    router.set_default(provider=rag_target)

    rag_provider, routed_rag_req = router.choose(req)

    try:
        rag_resp = rag_provider.chat(routed_rag_req)
        print(f"Wintermute (RAG Augmented): {rag_resp.content}")
    except Exception as e:
        print(f"RAG Query Failed: {e}")


test_dynamic_rag()

### How `set_default()` Influences Queries

Here is what happens at each stage of the example above:

**Step 1 — `init_router()`** registers all providers (Bedrock, Groq, OpenAI, HuggingFace)
and calls `bootstrap_rags(llms)`, which scans `knowledge_bases/` for subdirectories
containing a `storage_db/`. Each one becomes a `rag-<name>` provider in the `llms` registry.
The router defaults to `"bedrock"`.

**Step 2 — `router.set_default(provider="bedrock")`** explicitly sets the router's
`default_provider` to `"bedrock"`. When `router.choose(req)` is called, it returns the
raw Bedrock provider. The query goes directly to the LLM with no document context — so
the model can only answer from its general training data. For domain-specific questions
(like "Wintermute Quantum Processor pinout"), it will likely hallucinate or say it doesn't know.

**Step 3 — `router.set_default(provider="rag-tiny_hardware_test")`** switches the router
to the RAG provider. Now `router.choose(req)` returns the `RAGProvider` instance.
When `.chat()` is called on it, the RAG provider:
1. Queries the LlamaIndex vector store for chunks relevant to the user's question
2. Builds an augmented prompt: retrieved context + original query
3. Sends the augmented prompt to its `base_provider` (Bedrock)
4. The LLM now has the actual datasheet content in its context window

The key insight: **the same `ChatRequest` produces different answers depending on which
provider the router points to**. RAG providers transparently inject document context
before forwarding to the base LLM.

## Tools JSON — Linking Tools to RAG and MCP Backends

When RAG knowledge bases inform the AI about domain-specific tools (scripts, binaries,
hardware utilities), the AI needs to know **where those tools live on the filesystem**
to generate correct invocations. This is where the **tools JSON** comes in.

The `ToolRegistry.load_tool_configs()` method reads a JSON file that maps tool names to
their installation paths. When a tool is registered (either from Python functions via
`register_tools()`, or from MCP servers), the registry checks this mapping and enriches
the tool's description with its absolute filesystem path.

This is critical because:
- RAG documents may reference tools by name (e.g., "use `openocd` for JTAG access")
- The AI needs the full path to generate correct shell commands
- MCP tools from remote servers need local path resolution
- Different deployments install tools in different locations

### Tools JSON Format

The JSON is an **array of objects**, each mapping a tool name to its directory and executable:

```json
[
  {
    "name": "<tool_name>",
    "directory": "<relative_dir>",
    "executable": "<binary_or_script>"
  }
]
```

| Field | Description | Example |
|-------|-------------|--------|
| `name` | Tool name — must match the registered tool's name | `"openocd"` |
| `directory` | Subdirectory under `WINTERMUTE_TOOLS_ROOT` (default: `/opt`) | `"openocd/bin"` |
| `executable` | Binary or script filename | `"openocd"` |

The absolute path is resolved as: `{WINTERMUTE_TOOLS_ROOT}/{directory}/{executable}`

The `WINTERMUTE_TOOLS_ROOT` environment variable overrides the default base path (`/opt`).
This lets you adapt the same tools JSON to different environments without editing it.

In [ ]:
import json

# Example tools.json for a hardware security RAG knowledge base.
# These entries map tool names to their install locations on the host.
tools_config = [
    {
        "name": "openocd",
        "directory": "openocd/bin",
        "executable": "openocd",
    },
    {
        "name": "flashrom",
        "directory": "flashrom/bin",
        "executable": "flashrom",
    },
    {
        "name": "depthcharge",
        "directory": "depthcharge/bin",
        "executable": "depthcharge-inspect",
    },
    {
        "name": "tpm2_tools",
        "directory": "tpm2-tools/bin",
        "executable": "tpm2_getcap",
    },
]

print("Example tools.json:")
print(json.dumps(tools_config, indent=2))
print()
print("With WINTERMUTE_TOOLS_ROOT=/opt (default), resolved paths:")
for t in tools_config:
    print(f"  {t['name']} -> /opt/{t['directory']}/{t['executable']}")

### Using tools.json with the ToolRegistry

Below is a complete example showing how `load_tool_configs()` enriches registered tools.
When the AI sees a tool's description, it includes the absolute path — so LLM-generated
commands reference the correct binary location.

In [ ]:
import json
import os
import tempfile

from wintermute.ai.json_types import JSONObject
from wintermute.ai.tools_runtime import Tool, ToolRegistry

# 1. Create a tools.json config file
tools_config = [
    {
        "name": "openocd",
        "directory": "openocd/bin",
        "executable": "openocd",
    },
    {
        "name": "flashrom",
        "directory": "flashrom/bin",
        "executable": "flashrom",
    },
]

with tempfile.TemporaryDirectory() as tmpdir:
    tools_json_path = os.path.join(tmpdir, "tools.json")
    with open(tools_json_path, "w") as f:
        json.dump(tools_config, f)

    # 2. Create a registry with a custom base path and load the config
    registry = ToolRegistry(base_path="/opt/wintermute")
    registry.load_tool_configs(tools_json_path)

    # 3. Register tools — the registry enriches descriptions with absolute paths
    def openocd_handler(args: JSONObject) -> JSONObject:
        return {"result": f"Running OpenOCD with args: {args}"}

    def flashrom_handler(args: JSONObject) -> JSONObject:
        return {"result": f"Running flashrom with args: {args}"}

    openocd_tool = Tool(
        name="openocd",
        input_schema={"type": "object", "properties": {"config": {"type": "string"}}},
        output_schema={"type": "object"},
        handler=openocd_handler,
        description="Open On-Chip Debugger for JTAG/SWD access",
    )
    flashrom_tool = Tool(
        name="flashrom",
        input_schema={"type": "object", "properties": {"chip": {"type": "string"}}},
        output_schema={"type": "object"},
        handler=flashrom_handler,
        description="Flash ROM programmer for SPI/parallel flash",
    )

    registry.register(openocd_tool)
    registry.register(flashrom_tool)

    # 4. Inspect the enriched descriptions — absolute paths are now included
    print("Registered tools with path-enriched descriptions:\n")
    for defn in registry.get_definitions():
        fn = defn["function"]
        print(f"Tool: {fn['name']}")
        print(f"  Description: {fn['description']}")
        print(f"  Parameters: {fn['parameters']}")
        print()

### How Tools Connect to RAG and MCP at Runtime

The tools JSON, RAG providers, and MCP backends all converge at runtime through the
`ToolsRuntime` orchestrator:

```
ToolsRuntime.get_all_tools()
  ├─ ToolRegistry (static tools)
  │    ├─ Python functions via register_tools()
  │    ├─ MCP tools via MCPRuntime.initialize()
  │    └─ Path enrichment via load_tool_configs(tools.json)
  └─ Dynamic backends (e.g., SurgeonBackend)
       └─ MCP servers queried live via get_ai_tools()
```

When the AI is queried (e.g., `ai chat` in the console):
1. All tools are gathered from static + dynamic sources
2. The router selects the provider (could be a RAG provider or a base LLM)
3. Both are passed to `tool_calling_chat(router, messages, tools=tool_specs)`
4. If using a RAG provider, it augments the prompt with document context **and**
   forwards the `tools` list to the base LLM — the LLM sees both the RAG context
   and the available tools simultaneously

The tools JSON ensures that when the RAG context tells the AI about a tool (e.g., a
hardware datasheet mentions using `openocd`), the tool's description already contains
the correct absolute path for the current deployment.

## Using RAG in the Console

The Wintermute console (`WintermuteConsole`) provides built-in commands for RAG:

| Command | Description |
|---------|-------------|
| `ai rag list` | List all available knowledge bases |
| `ai rag use <name>` | Activate a knowledge base for AI queries |
| `ai rag off` | Deactivate RAG augmentation |
| `ai <question>` | Query using the active RAG (if any) |

When a RAG knowledge base is active, all `ai` queries are augmented with relevant
context retrieved from the indexed documents before being sent to the LLM.

## Summary

RAG knowledge bases in Wintermute:

1. **Create a directory** under `knowledge_bases/` with your documents and a `storage_db/` index
2. **Add `rag_config.json`** specifying the base LLM and embedding model
3. **Bootstrap** with `init_router()` or `bootstrap_rags()` to discover and register RAG providers
4. **Switch providers** with `router.set_default(provider="rag-<name>")` to enable RAG augmentation
5. **Link tools** with a `tools.json` via `ToolRegistry.load_tool_configs()` so the AI knows
   where tools referenced in RAG documents live on the filesystem
6. **Query** via the console (`ai rag use <name>`) or programmatically with `router.choose(req)`

Tips for effective knowledge bases:
- Use focused, domain-specific documents (datasheets, security advisories, standards)
- Smaller, well-structured documents index better than large monolithic files
- The `BAAI/bge-small-en-v1.5` embedding model is a good default for English text
- Pre-built indices in `storage_db/` are reused across sessions for fast startup
- Keep `tools.json` in sync with your deployment — use `WINTERMUTE_TOOLS_ROOT` to adapt paths